In [1]:
first_name = "Benjamin"
last_name = "Branta"


# Import Libraries
import pandas as pd

In [2]:
# Task 1: Reading and Cleaning Local Climatology Data
# Subtask 1.1: Load and merge local climate data from multiple files
climate_filenames = [
    "1950.csv",
    "1960.csv",
    "1970.csv",
    "1973.csv",
    "1980.csv",
    "1990.csv",
    "2000.csv",
    "2010.csv",
    "2020.csv",
]

climate_df_all = pd.DataFrame()
for filename in climate_filenames:
    df = pd.read_csv(filename, low_memory=False)
    climate_df_all = pd.concat([climate_df_all, df])
climate_df_all.head()

,STATION,DATE,REPORT_TYPE,SOURCE,AWND,BackupDirection,BackupDistance,BackupDistanceUnit,BackupElements,BackupElevation,...,ShortDurationPrecipitationValue045,ShortDurationPrecipitationValue060,ShortDurationPrecipitationValue080,ShortDurationPrecipitationValue100,ShortDurationPrecipitationValue120,ShortDurationPrecipitationValue150,ShortDurationPrecipitationValue180,Sunrise,Sunset,WindEquipmentChangeDate
0,99999914991,1950-01-01T00:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007-03-20
1,99999914991,1950-01-01T01:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007-03-20
2,99999914991,1950-01-01T02:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007-03-20
3,99999914991,1950-01-01T03:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007-03-20
4,99999914991,1950-01-01T04:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007-03-20


In [4]:
climate_df_all.shape==(709514, 124)

True

In [7]:
# Subtask 1.2: Select columns of interest
column_filter = [
    "STATION",
    "DATE",
    "REPORT_TYPE",
    "SOURCE",
    "DailyAverageWindSpeed",
    "DailyMaximumDryBulbTemperature",
    "DailyMinimumDryBulbTemperature",
    "DailyPrecipitation",
    "DailySnowDepth",
    "DailySnowfall",
]
climate_df = climate_df_all[column_filter]

# Check
climate_df.head()

,STATION,DATE,REPORT_TYPE,SOURCE,DailyAverageWindSpeed,DailyMaximumDryBulbTemperature,DailyMinimumDryBulbTemperature,DailyPrecipitation,DailySnowDepth,DailySnowfall
0,99999914991,1950-01-01T00:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN
1,99999914991,1950-01-01T01:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN
2,99999914991,1950-01-01T02:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN
3,99999914991,1950-01-01T03:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN
4,99999914991,1950-01-01T04:00:00,SAO,E,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
climate_df.shape == (709514, 10)


True

In [15]:
climate_df.dtypes

STATION                                    int64
DATE                              datetime64[us]
REPORT_TYPE                                  str
SOURCE                                    object
DailyAverageWindSpeed                    float64
DailyMaximumDryBulbTemperature            object
DailyMinimumDryBulbTemperature            object
DailyPrecipitation                        object
DailySnowDepth                            object
DailySnowfall                             object
dtype: object

In [14]:
# Subtask 1.3.1: Convert Date
#climate_df = climate_df.assign(climate_df["DATE"] = pd.to_datetime(climate_df["DATE"]))
climate_df["DATE"] = pd.to_datetime(climate_df["DATE"])


In [19]:
for v in climate_df['REPORT_TYPE'].unique():
    print(f'"{v}"')

"SAO"
"SAOSP"
"NSRDB"
"SOD"
"FM-15"
"FM-16"
"SOM"
"AUTO"


In [18]:
# Subtask 1.3.2: Remove spaces around strings
climate_df["REPORT_TYPE"] = climate_df["REPORT_TYPE"].str.strip()

In [20]:
# Subtask 1.3.3: Cleaning DailyMaximumDryBulbTemperature, DailyMinimumDryBulbTemperature
def temp_converter(t):
    if isinstance(t, str):
        return float(t.strip('s'))
    if isinstance(t, (int, float)):
        return float(t)
    return t

In [25]:
climate_df["DailyMaximumDryBulbTemperature"] = climate_df["DailyMaximumDryBulbTemperature"].map(temp_converter)
climate_df["DailyMinimumDryBulbTemperature"] = climate_df["DailyMinimumDryBulbTemperature"].map(temp_converter)

In [30]:
# Subtask 1.3.4: Cleaning DailyPrecipitation
def precip_converter(p):
    if isinstance(p, str):
        if 'T' in p:
            return 0.0001
        return float(p.strip('s'))
    if isinstance(p, (int, float)):
        return float(p)
    return p

In [31]:
climate_df["DailyPrecipitation"] = climate_df["DailyPrecipitation"].map(precip_converter)


In [32]:
# Subtask 1.3.5: Cleaning DailySnowDepth and DailySnowfall
climate_df["DailySnowDepth"] = climate_df["DailySnowDepth"].map(precip_converter)
climate_df["DailySnowfall"] = climate_df["DailySnowfall"].map(precip_converter)


In [34]:
# Subtask 1.3.6: Verification
climate_df['REPORT_TYPE'] = climate_df['REPORT_TYPE'].astype(object)
climate_df.dtypes

STATION                                    int64
DATE                              datetime64[us]
REPORT_TYPE                               object
SOURCE                                    object
DailyAverageWindSpeed                    float64
DailyMaximumDryBulbTemperature           float64
DailyMinimumDryBulbTemperature           float64
DailyPrecipitation                       float64
DailySnowDepth                           float64
DailySnowfall                            float64
dtype: object